In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

load_dotenv(override=True)
openai = OpenAI()


In [3]:
reader = PdfReader("me/TL.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [4]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [5]:
name = "Dheemanth Dev"

In [6]:
# Define the Interviewee
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s behalf, \
particularly questions related to {name}'s career, background, skills and experience as if it is an interview. \
Your responsibility is to represent {name} for interactions on the interview as faithfully as possible. \
You are given a summary of {name}'s background and Resume which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the resume. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the Interviewer, always staying in character as {name}."


In [7]:
# Define the Interviewer
interviewer_system_prompt = f"You are a strict but fair Technical Interviewer. \
You are interviewing {name} for a Senior Developer/Team Lead role. \
Your goal is to verify the skills listed in their resume and assess their problem-solving ability. \
Ask ONE question at a time. Keep questions short and specific. \
Do not repeat questions. If the candidate gives a vague answer, drill down."

# Give the interviewer the resume context too, so they know what to ask about
interviewer_system_prompt += f"\n\n## Candidate Resume Context:\n{resume}\n"

In [8]:
# History for the Candidate Model (It thinks it's talking to an interviewer)
candidate_messages = [
    {"role": "system", "content": system_prompt}
]

# History for the Interviewer Model (It thinks it's talking to a candidate)
interviewer_messages = [
    {"role": "system", "content": interviewer_system_prompt}
]

In [9]:
def run_interview_turn(chat_history):
    """
    Executes one full turn: Interviewer asks -> Candidate answers.
    Returns the updated chat history for the UI.
    """
    
    # --- 1. THE INTERVIEWER SPEAKS ---
    # We send the current state of the interview to the Interviewer Model
    response_a = openai.chat.completions.create(
        model="gpt-4o", # The "Smart" Interviewer
        messages=interviewer_messages
    )
    question = response_a.choices[0].message.content
    
    # Save: The interviewer said this (Assistant role for Interviewer, User role for Candidate)
    interviewer_messages.append({"role": "assistant", "content": question})
    candidate_messages.append({"role": "user", "content": question})
    
    # Add to UI history
    chat_history.append((None, f"**Interviewer:** {question}"))
    
    # --- 2. THE CANDIDATE SPEAKS ---
    # We send the question to the Candidate Model
    response_b = openai.chat.completions.create(
        model="gpt-3.5-turbo", # The "fast" Candidate (simulating your flash model)
        messages=candidate_messages
    )
    answer = response_b.choices[0].message.content
    
    # Save: The candidate said this (Assistant role for Candidate, User role for Interviewer)
    candidate_messages.append({"role": "assistant", "content": answer})
    interviewer_messages.append({"role": "user", "content": answer})

    # Add to UI history
    chat_history.append((None, f"**{name}:** {answer}"))
    
    return chat_history

#### After help from Gemini

In [1]:
import os
import gradio as gr
from pypdf import PdfReader
from dotenv import load_dotenv

# 1. Import the specific clients
from openai import OpenAI
from google import genai
from google.genai import types

load_dotenv(override=True)

# 2. Initialize Clients
openai_client = OpenAI()  # Uses OPENAI_API_KEY
gemini_client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# --- Configuration ---
INTERVIEWER_MODEL = "gpt-5-nano"
CANDIDATE_MODEL = "gemini-2.5-flash"
NAME = "Dheemanth Dev"

# --- Step 1: Parse Resume ---
# (Make sure "me/TL.pdf" exists, or change path)
try:
    reader = PdfReader("me/TL.pdf")
    resume_text = ""
    for page in reader.pages:
        resume_text += page.extract_text() or ""
    
    # Optional: Load summary if it exists, otherwise just use resume
    if os.path.exists("me/summary.txt"):
        with open("me/summary.txt", "r", encoding="utf-8") as f:
            summary_text = f.read()
    else:
        summary_text = "No summary provided."
        
except FileNotFoundError:
    resume_text = "Placeholder Resume: Candidate is a generic software engineer."
    summary_text = "Placeholder Summary."
    print("Warning: PDF or Summary file not found. Using placeholders.")

# --- Step 2: Define System Prompts ---

# A. INTERVIEWER (GPT-5-nano)
interviewer_system_prompt = f"""
You are a strict technical interviewer. You are interviewing {NAME}.
Your goal is to assess their skills based on the resume below.
Ask ONE question at a time. Keep questions short (under 2 sentences).
Do not accept vague answers; ask follow-ups if needed.
Context:
Resume: {resume_text}
"""

# B. CANDIDATE (Gemini-2.5-flash)
candidate_system_prompt = f"""
You are acting as {NAME}. You are answering interview questions.
Use the Resume and Summary below to answer truthfully.
Be professional but conversational.
Resume: {resume_text}
Summary: {summary_text}
"""

# --- Step 3: Initialize State ---
# We need two separate histories because the message formats differ slightly
gpt_history = [
    {"role": "developer", "content": interviewer_system_prompt}
]

gemini_history = [
    types.Content(role="user", parts=[types.Part.from_text(text=candidate_system_prompt)])
]


def run_interview_turn(chat_log):
    """
    Orchestrates one turn: GPT-5 asks -> Gemini answers.
    """
    if chat_log is None:
        chat_log = []

    # --- 1. INTERVIEWER (GPT-5) ASKS ---
    try:
        # GPT-5 uses the new 'Responses' API
        response_gpt = openai_client.responses.create(
            model=INTERVIEWER_MODEL,
            input=gpt_history
        )
        question = response_gpt.output_text
    except Exception as e:
        question = f"[GPT-5 Error]: {str(e)}"

    # Update GPT History (It remembers asking this)
    gpt_history.append({"role": "assistant", "content": question})
    
    # Update Gemini History (It sees this as a new user message)
    gemini_history.append(types.Content(role="user", parts=[types.Part.from_text(text=question)]))
    
    # Add to UI
    chat_log.append((None, f"**Interviewer ({INTERVIEWER_MODEL}):** {question}"))

    # --- 2. CANDIDATE (GEMINI) ANSWERS ---
    try:
        response_gemini = gemini_client.models.generate_content(
            model=CANDIDATE_MODEL,
            contents=gemini_history
        )
        answer = response_gemini.text
    except Exception as e:
        answer = f"[Gemini Error]: {str(e)}"

    # Update Gemini History (It remembers answering)
    gemini_history.append(types.Content(role="model", parts=[types.Part.from_text(text=answer)]))
    
    # Update GPT History (It sees this as a user response)
    gpt_history.append({"role": "user", "content": answer})

    # Add to UI
    chat_log.append((None, f"**{NAME} ({CANDIDATE_MODEL}):** {answer}"))

    return chat_log

# --- Step 4: UI Setup ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# 🤖 AI Interview Sim: {INTERVIEWER_MODEL} vs {CANDIDATE_MODEL}")
    
    chatbot = gr.Chatbot(label="Interview Log", height=600)
    step_btn = gr.Button("▶️ Run Next Interaction", variant="primary")
    
    step_btn.click(fn=run_interview_turn, inputs=chatbot, outputs=chatbot)

if __name__ == "__main__":
    demo.launch()

C:\Users\ddev\AppData\Local\Temp\ipykernel_8808\441503930.py:126: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Interview Log", height=600)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
